## Shivam Hippalgave

# Assignment - 6 part - 1

In [1]:
# 1. Install the MindSpore framework
!pip install mindspore

# 2. Clone the official MindSpore Lab models repository
# Remove any partially cloned 'models' directory before retrying
!rm -rf models
!git clone https://github.com/mindspore-lab/models.git

# 3. Navigate directly into the specific paper's directory
%cd models/research/arxiv_papers/miniformer

# 4. List the files to verify the structure
!ls -l

Cloning into 'models'...
remote: Enumerating objects: 9195, done.
remote: Counting objects: 100% (587/587), done.
remote: Compressing objects: 100% (220/220), done.
remote: Total 9195 (delta 474), reused 367 (delta 367), pack-reused 8608 (from 2)
Receiving objects: 100% (9195/9195), 367.77 MiB | 17.51 MiB/s, done.
Resolving deltas: 100% (3918/3918), done.
Updating files: 100% (8003/8003), done.
/content/models/research/arxiv_papers/miniformer
total 12
drwxr-xr-x 2 root root 4096 Apr 14 14:54 code
drwxr-xr-x 2 root root 4096 Apr 14 14:54 NPU_code
-rw-r--r-- 1 root root  746 Apr 14 14:54 README.md


In [2]:
!cat README.md

# README

code中为gpu上可以直接运行的代码，环境如下：

```
python==3.9.1
mindspore=2.2.14
mindnlp=0.3.1
```

NPU_code为可以在npu上运行的代码，环境一样，其余的训练测试等指令也一样。

数据集NMT14可从这里下载：https://aistudio.baidu.com/datasetdetail/1999

- 训练：

```
python train_seq2seqsum.py --data_path 训练集 --name 名称 --epoch 训练轮数
```

- 测试

进入代码，修改读入的模型和对应的测试集，以及最终的输出文件

```
python decode.py 
```

- 评估：

进入代码，修改模型的输出文件夹

```
python eval.py
```

- 对比：

对比的Transformer（包括数据集处理代码）可见：https://github.com/Veteranback/transformer


In [3]:
!pip install mindspore==2.2.14
!pip install mindnlp==0.3.1

ERROR: Could not find a version that satisfies the requirement mindspore==2.2.14 (from versions: 2.7.2, 2.8.0)
ERROR: No matching distribution found for mindspore==2.2.14
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.8/316.8 kB 34.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 530.0/530.0 kB 49.4 MB/s eta 0:00:00
  Attempting uninstall: pytest
    Found existing installation: pytest 8.4.2
    Uninstalling pytest-8.4.2:
      Successfully uninstalled pytest-8.4.2


In [26]:
import os
import pickle

vocab_dir = 'WMT14/raw'
os.makedirs(vocab_dir, exist_ok=True)

# 1. Create a vocabulary dictionary containing standard Seq2Seq special tokens
# 2. Add the numbers from your dummy text files so the model recognizes them
dummy_vocab = {
    '<pad>': 0,
    '<s>': 1,      # Start of sequence (Fixes your KeyError)
    '</s>': 2,     # End of sequence
    '<unk>': 3,    # Unknown token
    '1': 4, '2': 5, '3': 6, '4': 7, '5': 8, '6': 9,
    '7': 10, '8': 11, '9': 12, '10': 13, '11': 14, '12': 15,
    '13': 16, '14': 17, '15': 18, '16': 19, '17': 20, '18': 21
}

# Save the properly populated vocabulary
with open(os.path.join(vocab_dir, 'src_vocab.pkl'), 'wb') as f:
    pickle.dump(dummy_vocab, f)

with open(os.path.join(vocab_dir, 'tgt_vocab.pkl'), 'wb') as f:
    pickle.dump(dummy_vocab, f)

print("Vocabulary files successfully updated with special tokens!")

Vocabulary files successfully updated with special tokens!


In [27]:
!python code/train_seq2seqsum.py --data_path ./dummy_data --name assignment_run --epoch 2

[WARNING] ME(9034:138779604771456,MainProcess):2026-04-14-15:07:39.164.000 [mindspore/nn/layer/rnn_cells.py:68] LSTMCell has been changed from 'single LSTM layer' to 'single LSTM cell', if you still need use single LSTM layer, please use `nn.LSTM` instead.
[WARNING] ME(9034:138779604771456,MainProcess):2026-04-14-15:07:39.295.000 [mindspore/common/_decorator.py:69] 'FusedSparseAdam' is deprecated from version 2.8.0 and will be removed in a future version.
start val
Epoch 1, validation average loss: 3.0695
start val
Epoch 2, validation average loss: 3.0695


In [38]:
import glob

# 1. Search for any generated checkpoint files in the directory
ckpt_files = glob.glob('./**/*.ckpt', recursive=True)

if not ckpt_files:
    print("No checkpoint found. Please make sure the training cell finished successfully.")
else:
    # Get the actual path of the model we just trained
    actual_ckpt_path = ckpt_files[-1] # Grabs the latest checkpoint
    print(f"Found your trained checkpoint at: {actual_ckpt_path}")

    # 2. Read the current decode.py file
    with open('code/decode.py', 'r') as f:
        decode_code = f.read()

    # 3. Replace the hardcoded path with your actual path
    decode_code = decode_code.replace('"./model_WMT14/ckpt-6e-0s.ckpt"', f'"{actual_ckpt_path}"')
    decode_code = decode_code.replace("'./model_WMT14/ckpt-6e-0s.ckpt'", f'"{actual_ckpt_path}"')

    # 4. Write the fixed code back to the file
    with open('code/decode.py', 'w') as f:
        f.write(decode_code)

    print("Successfully patched decode.py!")

# 5. Run the decode script again
!python code/decode.py

Found your trained checkpoint at: ./model_assignment_run/ckpt-1e-0s.ckpt
Successfully patched decode.py!
[WARNING] ME(11511:133529566380672,MainProcess):2026-04-14-15:16:57.606.000 [mindspore/nn/layer/rnn_cells.py:68] LSTMCell has been changed from 'single LSTM layer' to 'single LSTM cell', if you still need use single LSTM layer, please use `nn.LSTM` instead.
1 pred:[] 
tgt:['13', '14', '15']



In [39]:
!python /content/models/research/arxiv_papers/miniformer/code/eval.py